In [15]:
%pip install pykrx pandas
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [242]:
import os

import time
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(".env", override=True)
from pykrx import stock
from dotenv import load_dotenv

load_dotenv()

True

In [19]:
# kospi index
def collect_index_future_returns(
    start_date: str,
    end_date: str,
    ticker: str,
    index_name: str,
    output_path: str,
):
    """지수 종가와 t+1~t+5 거래일의 미래 종가 수익률을 Excel로 저장합니다."""
    df = stock.get_index_ohlcv(
        start_date, end_date, ticker, name_display=False
    )

    if df.empty or '종가' not in df.columns:
        raise RuntimeError(f'{index_name} 지수 데이터를 받지 못했습니다.')

    result = pd.DataFrame(index=df.index)
    result['Index_Code'] = ticker
    result['Index_Name'] = index_name
    result['Close'] = df['종가']
    result['Daily_Return'] = result['Close'].pct_change()

    for horizon in range(1, 6):
        future_close = result['Close'].shift(-horizon)
        result[f'Close_t_plus_{horizon}'] = future_close
        result[f'Future_Return_t_plus_{horizon}'] = (
            future_close / result['Close'] - 1
        )

    result.index.name = 'Date'
    result.reset_index(inplace=True)
    result.to_excel(output_path, index=False, engine='openpyxl')
    print(f'{index_name} 지수 수익률 저장 완료: {output_path} ({len(result)}일)')
    return result


kospi_index_returns = collect_index_future_returns(
    start_date='20210701',
    end_date='20260630',
    ticker='1001',
    index_name='KOSPI',
    output_path='./kospi_index_returns.xlsx',
)

KRX 세션 만료, 재로그인 시도...
KRX 세션 갱신 완료.
KOSPI 지수 수익률 저장 완료: ./kospi_index_returns.xlsx (1222일)


In [4]:
# kosdaq index
kosdaq_index_returns = collect_index_future_returns(
    start_date='20210701',
    end_date='20260630',
    ticker='2001',
    index_name='KOSDAQ',
    output_path='./kosdaq_index_returns.xlsx',
)

KOSDAQ 지수 수익률 저장 완료: ./kosdaq_index_returns.xlsx (1222일)


In [2]:
# kospi by ticker

def collect_kospi_daily_data(start_date: str, end_date: str, directory: str = "./data_kospi"):
    """KOSPI 시장의 일별 OHLCV 및 개인/기관/외국인 순매수 데이터 수집"""
    os.makedirs(directory, exist_ok=True)
    
    # 영업일 리스트 추출
    sample = stock.get_index_ohlcv(start_date, end_date, "1001", name_display=False)
    trading_days = sample.index.strftime("%Y%m%d").tolist()
    
    print(f"--- KOSPI 데이터 수집 시작 (총 {len(trading_days)}일) ---")
    
    for date in trading_days:
        file_path = os.path.join(directory, f"kospi_{date}.xlsx")
        if os.path.exists(file_path):
            continue
            
        try:
            # 1. KOSPI 전 종목 OHLCV 조회
            df_kospi = stock.get_market_ohlcv(date, market="KOSPI")
            
            # 2. 개인 및 전체 투자자의 종목별 매수·매도 주식 수 조회
            investor_prefixes = {
                '개인': 'Retail',
                '기관합계': 'Institution',
                '외국인': 'Foreign',
                '전체': 'Total',
            }
            volume_frames = []
            for investor, prefix in investor_prefixes.items():
                investor_df = stock.get_market_net_purchases_of_equities(
                    date, date, market="KOSPI", investor=investor
                )
                volume_frames.append(
                    investor_df[['매수거래량', '매도거래량']].rename(columns={
                        '매수거래량': f'{prefix}_Buy_Volume',
                        '매도거래량': f'{prefix}_Sell_Volume',
                    })
                )
            df_investor_volume = (
                pd.concat(volume_frames, axis=1).fillna(0).astype('int64')
            )
            
            # 3. 종목코드 인덱스를 기준으로 OHLCV와 투자자 거래량 병합
            df_merged = df_kospi.join(df_investor_volume, how='inner')
            
            # 4. 컬럼 정리 및 저장
            df_merged.insert(0, 'Date', date)
            df_merged.index.name = 'Ticker'
            df_merged.reset_index(inplace=True)
            
            df_merged.to_excel(file_path, index=False, engine='openpyxl')
            print(f"[KOSPI] {date} 수집 완료")
            time.sleep(0.5) # KRX 서버 차단 방지
            
        except Exception as e:
            print(f"[KOSPI 오류] {date}: {e}")

In [3]:
# kosdaq by ticker
def collect_kosdaq_daily_data(start_date: str, end_date: str, directory: str = "./data_kosdaq"):
    """KOSDAQ 시장의 일별 OHLCV 및 개인/기관/외국인 순매수 데이터 수집"""
    os.makedirs(directory, exist_ok=True)
    
    # 영업일 리스트 추출
    sample = stock.get_index_ohlcv(start_date, end_date, "2001", name_display=False)
    trading_days = sample.index.strftime("%Y%m%d").tolist()
    
    print(f"\n--- KOSDAQ 데이터 수집 시작 (총 {len(trading_days)}일) ---")
    
    for date in trading_days:
        file_path = os.path.join(directory, f"kosdaq_{date}.xlsx")
        if os.path.exists(file_path):
            continue
            
        try:
            # 1. KOSDAQ 전 종목 OHLCV 조회
            df_kosdaq = stock.get_market_ohlcv(date, market="KOSDAQ")
            
            # 2. 개인 및 전체 투자자의 종목별 매수·매도 주식 수 조회
            investor_prefixes = {
                '개인': 'Retail',
                '기관합계': 'Institution',
                '외국인': 'Foreign',
                '전체': 'Total',
            }
            volume_frames = []
            for investor, prefix in investor_prefixes.items():
                investor_df = stock.get_market_net_purchases_of_equities(
                    date, date, market="KOSDAQ", investor=investor
                )
                volume_frames.append(
                    investor_df[['매수거래량', '매도거래량']].rename(columns={
                        '매수거래량': f'{prefix}_Buy_Volume',
                        '매도거래량': f'{prefix}_Sell_Volume',
                    })
                )
            df_investor_volume = (
                pd.concat(volume_frames, axis=1).fillna(0).astype('int64')
            )
            
            # 3. 종목코드 인덱스를 기준으로 OHLCV와 투자자 거래량 병합
            df_merged = df_kosdaq.join(df_investor_volume, how='inner')
            
            # 4. 컬럼 정리 및 저장
            df_merged.insert(0, 'Date', date)
            df_merged.index.name = 'Ticker'
            df_merged.reset_index(inplace=True)
            
            df_merged.to_excel(file_path, index=False, engine='openpyxl')
            print(f"[KOSDAQ] {date} 수집 완료")
            time.sleep(0.5) # KRX 서버 차단 방지
            
        except Exception as e:
            print(f"[KOSDAQ 오류] {date}: {e}")

In [246]:
if __name__ == "__main__":
    START = "20180611"
    END = "20180620"
    
    # KOSPI 수집
    collect_kospi_daily_data(START, END, directory="./data_kospi")
    
    # KOSDAQ 수집
    collect_kosdaq_daily_data(START, END, directory="./data_kosdaq")

--- KOSPI 데이터 수집 시작 (총 7일) ---
[KOSPI] 20180611 수집 완료
[KOSPI] 20180612 수집 완료
[KOSPI] 20180614 수집 완료
[KOSPI] 20180615 수집 완료
[KOSPI] 20180618 수집 완료
[KOSPI] 20180619 수집 완료
[KOSPI] 20180620 수집 완료

--- KOSDAQ 데이터 수집 시작 (총 7일) ---
[KOSDAQ] 20180611 수집 완료
[KOSDAQ] 20180612 수집 완료
[KOSDAQ] 20180614 수집 완료
[KOSDAQ] 20180615 수집 완료
[KOSDAQ] 20180618 수집 완료
[KOSDAQ] 20180619 수집 완료
[KOSDAQ] 20180620 수집 완료
